# 03 - Validacion
Control de calidad final sobre data/processed/suicidio_2023_limpio.csv.
Objetivo: confirmar con evidencia cuantificable que el dataset limpio es confiable,
y llenar la tabla 'antes/despues' de docs/quality_report.md con numeros reales.

In [ ]:
import pandas as pd
import sys
sys.path.append('../src')
from cleaning_utils import (
    load_dbf, normalize_columns, CATALOG_CANONICAL_COLUMNS, null_summary, duplicate_report,
    validate_dates, validate_codes_against_catalog, validate_cross_consistency
)


## 1. Cargar dataset ANTES (interim, filtrado sin limpiar) y DESPUES (processed, limpio)

In [ ]:
df_antes = pd.read_csv('../data/interim/suicidio_2023_filtrado.csv', encoding='utf-8', low_memory=False, dtype=str)
df_despues = pd.read_csv('../data/processed/suicidio_2023_limpio.csv', encoding='utf-8', low_memory=False, dtype=str)
print(f'ANTES (interim):    {len(df_antes):,} filas x {df_antes.shape[1]} columnas')
print(f'DESPUES (processed): {len(df_despues):,} filas x {df_despues.shape[1]} columnas')


## 2. Validacion de volumen
La limpieza NO debe alterar el numero de registros (solo corrige valores).

In [ ]:
assert len(df_antes) == len(df_despues), 'ALERTA: el volumen de registros cambio durante la limpieza'
print(f'Volumen consistente: {len(df_despues):,} registros en ambos datasets. OK.')


## 3. Re-validacion contra cifra oficial INEGI
Repetir la comparacion contra la Nota Tecnica EDR 2023 (~9,085 casos) sobre el dataset YA LIMPIO,
para confirmar que ninguna regla de limpieza altero el universo de estudio.

In [ ]:
cifra_oficial = 9085
diferencia = abs(len(df_despues) - cifra_oficial)
pct_diferencia = round(diferencia / cifra_oficial * 100, 2)
print(f'Conteo propio (dataset limpio): {len(df_despues):,}')
print(f'Cifra oficial INEGI: {cifra_oficial:,}')
print(f'Diferencia: {diferencia} casos ({pct_diferencia}%)')
print('Validado' if pct_diferencia < 2 else 'REVISAR: diferencia mayor al margen esperado')


## 4. Validacion de fechas
Reconstruir Anio_ocur+Mes_ocurr+Dia_ocurr y verificar que formen fechas de calendario validas.

In [ ]:
resultado_fechas = validate_dates(df_despues, 'Anio_ocur', 'Mes_ocurr', 'Dia_ocurr')
resultado_fechas


## 5. Validacion de codigos geograficos contra catalogo vigente
Verificar que Ent_ocurr y Mun_ocurr existan en el catalogo CATEMLDE23.dbf (sin codigos huerfanos).

In [ ]:
cat_geo = load_dbf('../data/raw/CATEMLDE23.dbf')
cat_geo = normalize_columns(cat_geo, canonical_names=CATALOG_CANONICAL_COLUMNS)
cat_entidades = cat_geo[(cat_geo['Cve_mun'] == '000') & (cat_geo['Cve_loc'] == '0000')]

resultado_geo = validate_codes_against_catalog(df_despues, 'Ent_ocurr', cat_entidades, 'Cve_ent')
print(f\"Codigos de entidad sin match en catalogo: {resultado_geo['n_codigos_huerfanos']}\")
print(f\"Registros afectados: {resultado_geo['n_registros_afectados']}\")
resultado_geo


## 6. Consistencia cruzada: Tipo_defun == 3 (suicidio) vs Causa_def (CIE-10 X60-X84)
Si el tipo de defuncion es suicidio, la causa CIE-10 deberia caer en el rango de lesiones autoinfligidas.

In [ ]:
def es_causa_suicidio(codigo):
    if pd.isna(codigo):
        return False
    codigo = str(codigo).strip()
    if len(codigo) < 3 or codigo[0] != 'X':
        return False
    try:
        num = int(codigo[1:3])
    except ValueError:
        return False
    return 60 <= num <= 84

resultado_consistencia = validate_cross_consistency(
    df_despues, col_a='Tipo_defun', valor_a='3',
    col_b='Causa_def', valores_b_validos=es_causa_suicidio
)
resultado_consistencia


## 7. Duplicados (re-verificacion post-limpieza)

In [ ]:
duplicate_report(df_despues)


## 8. Tabla antes/despues de nulos explicitos
Comparar cuantos NaN explicitos hay antes (interim, sin recodificar) vs despues (processed, con recodificacion aplicada).

In [ ]:
nulos_antes = null_summary(df_antes)
nulos_despues = null_summary(df_despues)

comparacion = pd.DataFrame({
    'pct_nulos_antes': nulos_antes['pct_nulos'],
    'pct_nulos_despues': nulos_despues.reindex(nulos_antes.index)['pct_nulos'],
})
comparacion['diferencia'] = comparacion['pct_nulos_despues'] - comparacion['pct_nulos_antes']
comparacion.sort_values('diferencia', ascending=False).head(15)


## 9. Resumen final de validacion
_Completar con los resultados reales de las celdas anteriores y trasladar a docs/quality_report.md_

In [ ]:
print('=== RESUMEN DE VALIDACION ===')
print(f'Volumen: {len(df_despues):,} registros (consistente antes/despues: {len(df_antes)==len(df_despues)})')
print(f\"Fechas invalidas: {resultado_fechas['fechas_invalidas']} ({resultado_fechas['pct_invalidas']}%)\")
print(f\"Codigos geograficos huerfanos: {resultado_geo['n_codigos_huerfanos']}\")
print(f\"Inconsistencias Tipo_defun vs Causa_def: {resultado_consistencia['n_inconsistentes']} ({resultado_consistencia['pct_inconsistentes']}%)\")
print(f'Diferencia vs cifra oficial INEGI: {diferencia} casos ({pct_diferencia}%)')
